In [2]:
import pandas as pd

from src.leakage import remove_leakage_columns
from src.target import add_target, get_resolve_loan_status
from src.test_set import get_test_set

df = pd.read_parquet("data/interim/accepted_2007_to_2018Q4.parquet")

df = df.drop(columns=["member_id", "id"])

df_without_leakage = remove_leakage_columns(df)

resolved = get_resolve_loan_status(df_without_leakage)
resolved_with_target = add_target(resolved)

train, test = get_test_set(resolved_with_target, "2016-10-01")

train.to_parquet("data/processed/train.parquet", index=False)
test.to_parquet("data/processed/test.parquet", index=False)


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

dummy_model = DummyClassifier(strategy="prior")

y_test = test["target"]
y_train = train["target"]
X_test = test.drop(columns=["target"])
X_train = train.drop(columns=["target"])

dummy_model.fit(X_train, y_train)

proba = dummy_model.predict_proba(X_test)[:, 1]

print(f"Dummy Model ROC AUC Score: {roc_auc_score(y_test, proba):.2%}")


Dummy Model ROC AUC Score: 50.00%


In [4]:
grade_mapping = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

encoded_grade = X_test["grade"].map(grade_mapping)
grade_raw_score = roc_auc_score(y_test, encoded_grade)
grade_auc = max(grade_raw_score, 1 - grade_raw_score)

fico_score = (X_test["fico_range_low"] + X_test["fico_range_high"]) / 2
fico_raw_score = roc_auc_score(y_test, fico_score)
fico_auc = max(fico_raw_score, 1 - fico_raw_score)

print(f"Grade Baseline ROC AUC Score: {grade_auc:.2%}")
print(f"FICO Baseline ROC AUC Score: {fico_auc:.2%}")


Grade Baseline ROC AUC Score: 66.97%
FICO Baseline ROC AUC Score: 59.97%


# Understanding Week 3

## Functions

To better automate certain tasks, functions have been added to streamline some processes.

## Test and Training Data

`get_test_set` return both test and training data based on a date input. Both test and training data have been save under processed data for later use.

## Dummy Model

The dummy model ROC AUC is 50%. This is because it never looks at any feature. This is the floor of our model. If it performs worst than the dummy, we have failed.

## Grade and FICO Baseline

The grade baseline ROC AUC is 66.97%, while FICO baseline ROC AUC is 59.97%. Both beat the dummy model, and we will use the grade baseline for our model to beat. However, compared to our ROC AUC from week 2 we can see that we its only ~5.6% above. This means that the grading are responsible for a lot for our models high AUC score already.
